# Unit 3

## Handling AI Interactions with the Chat Service Layer

## 🤖 Handling AI Interactions with the Chat Service Layer

In the previous lesson, we explored the **ChatManager** class, which plays a crucial role in managing chat data within our application. Now, we will take the next step in our journey by building the **Chat Service Layer**. This layer is essential for integrating the language model with chat sessions, allowing us to process user messages and generate AI responses. By the end of this lesson, you will understand how to set up the **ChatService** class, create chat sessions, and process messages using OpenAI's API.

-----

The **service layer** acts as a bridge between the model layer, where data is managed, and the AI model, which generates responses. It is responsible for orchestrating the flow of data and ensuring that user interactions are handled smoothly. Let's dive into the details of setting up this important component.

## Setting Up the ChatService Class

The **ChatService** class is the heart of our service layer. It is responsible for managing chat sessions and interacting with the OpenAI client to generate AI responses. To begin, we need to set up the class and its components.

First, we import the necessary modules, including the **ChatManager** from our previous lesson and the **OpenAI** client. We also use the **uuid** module to generate unique chat IDs. Here's how the class is initialized:

```python
import uuid
from openai import OpenAI
from models.chat import ChatManager

class ChatService:
    def __init__(self):
        self.chat_manager = ChatManager()
        self.openai_client = OpenAI()
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
```

In this setup, we instantiate `ChatManager` to manage chat data, initialize the **OpenAI client**, and load the `system_prompt` using the `load_system_prompt` method, which we'll discuss next.

-----

## Loading the System Prompt

The **system prompt** is a crucial component that guides the AI's responses. It provides context and instructions for the AI, ensuring that it behaves in a manner consistent with our application's goals. In this section, we'll implement the `load_system_prompt` method to load the prompt from a file.

```python
def load_system_prompt(self, file_path: str) -> str:
    """Load the system prompt from file."""
    try:
        with open(file_path, 'r') as f:
            return f.read()
    except Exception as e:
        print(f"Error loading system prompt: {e}")
        return "You are a helpful assistant."
```

This method attempts to read the system prompt from a specified file path. If successful, it returns the prompt as a string. In case of an error, it prints an error message and returns a **default prompt**. This ensures that the application can continue functioning even if the file is missing or corrupted.

-----

## Creating a New Chat Session

Creating a new chat session is a fundamental task of the `ChatService`. The `create_chat` method is responsible for generating a unique chat ID and initializing a chat session using the `ChatManager`.

```python
def create_chat(self, user_id: str) -> str:
    """Create a new chat session."""
    chat_id = str(uuid.uuid4())
    self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
    return chat_id
```

In this method, we generate a unique `chat_id` using the **uuid** module. We then call the `create_chat` method of `ChatManager`, passing the `user_id`, `chat_id`, and `system_prompt`. This initializes a new chat session, which is ready to receive messages.

-----

## Processing User Messages

The `process_message` method is where the magic happens. It processes user messages, interacts with the OpenAI client to generate AI responses, and updates the chat history. Below, we outline the steps involved in this process, followed by the corresponding code implementation:

1.  **Retrieve** the chat using `get_chat`, and raise an error if the chat is not found.
2.  **Add** the user's message to the chat history.
3.  **Send** the conversation, including the system prompt and all messages, to the OpenAI client to generate a response.
4.  **Add** the AI's response to the chat history and return it to the user.
5.  **Handle** any errors with the AI client gracefully.

<!-- end list -->

```python
def process_message(self, user_id: str, chat_id: str, message: str) -> str:
    """Process a user message and get AI response."""
    # Step 1: Retrieve the chat
    chat = self.chat_manager.get_chat(user_id, chat_id)
    if not chat:
        raise ValueError("Chat not found")

    # Step 2: Add user message to chat history
    self.chat_manager.add_message(user_id, chat_id, "user", message)

    try:
        # Step 3: Get AI response
        conversation = self.chat_manager.get_conversation(user_id, chat_id)

        response = self.openai_client.chat.completions.create(
            model="gpt-4",
            messages=conversation,
            temperature=0.7,
            max_tokens=500
        )

        ai_message = response.choices[0].message.content
        
        # Step 4: Add AI response to chat history
        self.chat_manager.add_message(user_id, chat_id, "assistant", ai_message)

        return ai_message

    except Exception as e:
        # Step 5: Handle errors
        raise RuntimeError(f"Error getting AI response: {str(e)}")
```

In the context of a customer service agent, we configure our model with specific parameters to optimize its performance. The **`temperature`** is set to **0.7**, which balances creativity and coherence in the AI's responses, ensuring they are both engaging and relevant. The **`max_tokens`** is set to **500**, allowing the model to provide detailed and informative answers without overwhelming the user, thus maintaining a smooth and effective customer service experience.

-----

## Example: Simulating a Chat Session

Let's see the `ChatService` in action by simulating a chat session. We'll use the `main.py` file to create a chat session and process a user message.

```python
from services.chat_service import ChatService

# Initialize the chat service
chat_service = ChatService()

# Simulate a user ID
user_id = "user123"

# Create a new chat session
chat_id = chat_service.create_chat(user_id)
print(f"Chat session created with ID: {chat_id}")

# Simulate sending a message
user_message = "Hello, how are you?"
try:
    ai_response = chat_service.process_message(user_id, chat_id, user_message)
    print(f"AI Response: {ai_response}")
except Exception as e:
    print(f"Error: {e}")
```

In this example, we initialize the `ChatService`, simulate a user ID, and create a new chat session, printing the chat ID. We then simulate sending a message and print the AI's response, demonstrating the flow from user input to AI response and showcasing the functionality of the `ChatService`.

```text
Chat session created with ID: 01a17870-8a4f-4b6f-a3ce-f04e1136d597
AI Response: Hello! I'm here to help with any questions or concerns you might have regarding our IT services. How can I assist you today?
```

This output illustrates a successful interaction where a new chat session is created, and the AI responds to the user's greeting with a helpful message. The AI's response is tailored to assist with IT services, showcasing the system's ability to provide relevant and context-aware assistance.

-----

## Summary and Next Steps

In this lesson, we explored the **ChatService** class and its role in integrating the language model with chat sessions. We learned how to set up the class, load the system prompt, create chat sessions, and process user messages. The **service layer** is a vital component of our chatbot application, ensuring that user interactions are handled smoothly and efficiently.

As you move on to the practice exercises, take the opportunity to experiment with the `ChatService` functionality. This hands-on practice will reinforce the concepts covered in this lesson and prepare you for the next steps in our course. Keep up the great work, and I look forward to seeing your progress\!

## Creating Unique Chat Sessions

Let's begin by implementing the create_chat method within the ChatService class. This method is essential for generating a unique chat ID and initializing a chat session. Here's what you need to do:

Implement the create_chat method within the ChatService class.

Use the uuid module to generate a unique chat ID.
Utilize the ChatManager to create a chat session, associating it with the user ID, chat ID, and system prompt.
Ensure the method returns the generated chat ID.
Test the chat creation using the service.

Initialize an instance of ChatService.
Define a sample user ID to simulate a user.
Call the create_chat method with the user ID and store the returned chat ID.
Print the chat ID to verify the successful creation of the chat session.
This task will guide you through the process of creating and managing chat sessions, a fundamental aspect of building a chatbot service.


```python
import uuid
from openai import OpenAI
from models.chat import ChatManager

class ChatService:
    def __init__(self):
        # TODO: Initialize ChatManager to handle chat data
        self.chat_manager = 
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file."""
        try:
            with open(file_path, 'r') as f:
                return f.read()
        except Exception as e:
            print(f"Error loading system prompt: {e}")
            return "You are a helpful assistant."
    
    # TODO: Define the create_chat method
    # - Parameters: user_id
    # - Generate a unique chat ID using uuid
    # - Use chat_manager to create a chat session with user_id, chat_id, and system_prompt
    # - Return the chat_id

    from services.chat_service import ChatService

# TODO: Initialize a ChatService instance

# TODO: Define a variable with a sample user identifier, e.g., "user123"

# TODO: Call the create_chat method with the user ID and store the chat ID

# TODO: Print the chat ID

```


That's a great task for solidifying the service layer\! Since the `ChatManager` and `OpenAI` classes aren't fully defined here, I will provide the completed code block assuming the `ChatManager` class from the previous context is available and simplifying the `load_system_prompt` for execution.

Here is the completed code for the `ChatService` class and the test script:

```python
import uuid
from openai import OpenAI  # Assuming this is available
# from models.chat import ChatManager  # Assuming this is defined elsewhere

# Re-defining a simplified ChatManager for a runnable example
class ChatManager:
    def __init__(self):
        self.chats = {} # user_id -> chat_id -> chat_data
    
    def create_chat(self, user_id, chat_id, system_prompt):
        """Create a new chat for a user."""
        if user_id not in self.chats:
            self.chats[user_id] = {}
        
        self.chats[user_id][chat_id] = {
            'system_prompt': system_prompt,
            'messages': []
        }
    
    # Placeholder for other methods
    def get_chat(self, user_id, chat_id):
        return self.chats.get(user_id, {}).get(chat_id)
    def add_message(self, user_id, chat_id, role, content):
        pass # Not needed for this exercise

class ChatService:
    def __init__(self):
        # ✅ TODO: Initialize ChatManager to handle chat data
        self.chat_manager = ChatManager()
        # self.openai_client = OpenAI() # Not needed for this specific method
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file. (Simplified for example)"""
        # Original logic remains, but we return a default since we can't read a file
        return "You are a helpful assistant."
        # try:
        #     with open(file_path, 'r') as f:
        #         return f.read()
        # except Exception as e:
        #     print(f"Error loading system prompt: {e}")
        #     return "You are a helpful assistant."
    
    # ✅ TODO: Define the create_chat method
    def create_chat(self, user_id: str) -> str:
        """Create a new chat session and return its unique ID."""
        
        # - Generate a unique chat ID using uuid
        chat_id = str(uuid.uuid4())
        
        # - Use chat_manager to create a chat session with user_id, chat_id, and system_prompt
        self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
        
        # - Return the chat_id
        return chat_id


# --- Testing the Chat Creation ---
print("## 🚀 Testing Chat Session Creation")

# ✅ TODO: Initialize a ChatService instance
chat_service = ChatService()

# ✅ TODO: Define a variable with a sample user identifier, e.g., "user123"
sample_user_id = "user123"
print(f"User ID: **{sample_user_id}**")

# ✅ TODO: Call the create_chat method with the user ID and store the chat ID
new_chat_id = chat_service.create_chat(sample_user_id)

# ✅ TODO: Print the chat ID
print(f"**Successfully created new chat session!**")
print(f"Generated Chat ID: **{new_chat_id}**")

# Optional: Verify it exists in the manager (using the placeholder get_chat)
chat_data = chat_service.chat_manager.get_chat(sample_user_id, new_chat_id)
print(f"Verification: Chat session found with system prompt: '{chat_data['system_prompt']}'")
```

### Key Implementation Details

  * **Unique ID:** `chat_id = str(uuid.uuid4())` uses the `uuid` module to generate a universally unique identifier, which is critical for distinguishing one chat session from another. We convert it to a string for easy use.
  * **ChatManager Integration:** `self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)` seamlessly passes the necessary data to the data management layer (`ChatManager`) to officially record the new chat.
  * **Return Value:** The method returns `chat_id`, which is essential for all subsequent message processing steps in this chat session.

Would you like to move on to implementing the **`process_message`** method, which handles sending messages to the AI?

## Implementing the Process Message Method

You've done well setting up the ChatService class and creating chat sessions. Now, let's begin implementing the process_message method to handle user messages. For now, we will focus on storing the user's message without getting a response from the AI. Here's what you need to do:

Implement the process_message method within the ChatService class.

Check if the chat session exists using the get_chat method from ChatManager.
If the chat doesn't exist, raise a ValueError with the message "Chat not found".
Add the user's message to the chat history using the add_message method from ChatManager.
Return a string indicating the message was processed, e.g., "Message processed".
Test the process_message method by simulating a user message, ensuring the chat history is updated correctly, and handle any exceptions by printing them if the chat session does not exist.

This task will ensure that your chat service can effectively handle user messages, setting the stage for integrating AI responses in the future.



```python
import uuid
from openai import OpenAI
from models.chat import ChatManager

class ChatService:
    def __init__(self):
        self.chat_manager = ChatManager()
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file."""
        try:
            with open(file_path, 'r') as f:
                return f.read()
        except Exception as e:
            print(f"Error loading system prompt: {e}")
            return "You are a helpful assistant."
    
    def create_chat(self, user_id: str) -> str:
        """Create a new chat session."""
        chat_id = str(uuid.uuid4())
        self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
        return chat_id
    
    # TODO: Define the process_message method
    # - Parameters: user_id, chat_id, message
    # - Check if the chat session exists using get_chat
    #   - If the chat doesn't exist, raise a ValueError with "Chat not found"
    # - Add the user's message to the chat history using add_message
    # - Return a string indicating the message was processed

```

Great\! Implementing the message processing logic is a core step in building the service.

Since your prompt focuses on the implementation and testing of `process_message` without AI integration, I'll complete the `ChatService` and provide a runnable test scenario. I will include the necessary `ChatManager` definition for a complete, executable example.

### 🐍 Completed ChatService and Test Script

```python
import uuid
# from openai import OpenAI # Not needed for this step

# --- Mocking necessary external classes for execution ---
class ChatManager:
    def __init__(self):
        self.chats = {}  # user_id -> chat_id -> chat_data
    
    def create_chat(self, user_id, chat_id, system_prompt):
        """Create a new chat for a user."""
        if user_id not in self.chats:
            self.chats[user_id] = {}
        
        self.chats[user_id][chat_id] = {
            'system_prompt': system_prompt,
            'messages': []
        }
    
    def get_chat(self, user_id, chat_id):
        """Get a chat by user_id and chat_id."""
        return self.chats.get(user_id, {}).get(chat_id)
    
    def add_message(self, user_id, chat_id, role, content):
        """Add a message to a chat."""
        if chat := self.get_chat(user_id, chat_id):
            chat['messages'].append({"role": role, "content": content})
    
    def get_conversation(self, user_id, chat_id):
        """Get the full conversation including system message."""
        if chat := self.get_chat(user_id, chat_id):
            return [
                {"role": "system", "content": chat['system_prompt']}
            ] + chat['messages']
        return []

# --- ChatService Implementation ---
class ChatService:
    def __init__(self):
        self.chat_manager = ChatManager()
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file. (Simplified for this example)"""
        return "You are a helpful assistant."
    
    def create_chat(self, user_id: str) -> str:
        """Create a new chat session."""
        chat_id = str(uuid.uuid4())
        self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
        return chat_id
    
    # ✅ TODO: Define the process_message method
    def process_message(self, user_id: str, chat_id: str, message: str) -> str:
        """Process a user message and store it in chat history."""
        
        # Check if the chat session exists using get_chat
        chat = self.chat_manager.get_chat(user_id, chat_id)
        
        # If the chat doesn't exist, raise a ValueError with "Chat not found"
        if not chat:
            raise ValueError("Chat not found")
        
        # Add the user's message to the chat history using add_message
        self.chat_manager.add_message(user_id, chat_id, "user", message)
        
        # Return a string indicating the message was processed
        return "Message processed"

# --- Testing the process_message method ---
print("## ✨ Testing Message Processing")
chat_service = ChatService()
user_id = "test_user_456"
test_message = "What is the status of my order?"

# 1. Test Case: Successful message processing
print("\n--- Test 1: Successful Message Processing ---")
try:
    # Create a valid chat session
    valid_chat_id = chat_service.create_chat(user_id)
    print(f"Chat created with ID: {valid_chat_id}")

    # Process the message
    result = chat_service.process_message(user_id, valid_chat_id, test_message)
    print(f"Result: {result}")
    
    # Verify history update
    conversation = chat_service.chat_manager.get_conversation(user_id, valid_chat_id)
    last_message = conversation[-1]

    print("Verification:")
    print(f"  Total messages (including system prompt): {len(conversation)}")
    print(f"  Last Message Role: **{last_message['role']}**")
    print(f"  Last Message Content: **{last_message['content']}**")
    
except Exception as e:
    print(f"An unexpected error occurred in Test 1: {e}")

# 2. Test Case: Chat Not Found Exception Handling
print("\n--- Test 2: Chat Not Found Error Handling ---")
invalid_chat_id = "non-existent-chat-123"
try:
    print(f"Attempting to process message with invalid ID: {invalid_chat_id}")
    chat_service.process_message(user_id, invalid_chat_id, test_message)
except ValueError as e:
    print(f"Successfully caught expected error: **{e}**")
except Exception as e:
    print(f"Caught an incorrect error type: {e}")
```

### Key Implementation Points:

1.  **Chat Existence Check:**

    ```python
    chat = self.chat_manager.get_chat(user_id, chat_id)
    if not chat:
        raise ValueError("Chat not found")
    ```

    This is a critical **guard clause**. It ensures that all subsequent operations only happen if a valid chat context exists, preventing errors in the `ChatManager`.

2.  **Message Storage:**

    ```python
    self.chat_manager.add_message(user_id, chat_id, "user", message)
    ```

    The user's message is correctly added to the chat history with the `role` set to `"user"`.

3.  **Testing:** The test scenario successfully verifies two conditions:

      * A message is processed, and the **chat history is updated** (Test 1).
      * A `ValueError` is correctly raised and caught when a **non-existent `chat_id`** is used (Test 2).

The service is now ready for the next step: integrating the AI response logic\!

## Integrate AI Responses in Chat

Nice work on setting up the process_message method! Now, let's enhance it to generate AI responses. Your task is to add the missing code that interacts with the OpenAI client to obtain the AI response and update the chat history.

Here's what you need to do in the ChatService class:

Instantiate the OpenAI client to interact with the AI model.
Retrieve the conversation using get_conversation from ChatManager.
Use the OpenAI client to generate a response from "gpt-4" based on the conversation, set the temperature to 0.7 and the maximum tokens to 500 to balance creativity and coherence in responses.
Extract the AI's response from the OpenAI client response.
Add the AI's response to the chat history.
Return the AI's response.
Remember to handle exceptions during the OpenAI client interaction by raising a RuntimeError with an appropriate message.

The main code is pre-configured, allowing you to test the integration seamlessly. Keep going; you're doing well!

```python
#chat_service.py
import uuid
from openai import OpenAI
from models.chat import ChatManager

class ChatService:
    def __init__(self):
        self.chat_manager = ChatManager()
        # TODO: Initialize the OpenAI client to interact with the AI model
        self.openai_client = 
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file."""
        try:
            with open(file_path, 'r') as f:
                return f.read()
        except Exception as e:
            print(f"Error loading system prompt: {e}")
            return "You are a helpful assistant."
    
    def create_chat(self, user_id: str) -> str:
        """Create a new chat session."""
        chat_id = str(uuid.uuid4())
        self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
        return chat_id
    
    def process_message(self, user_id: str, chat_id: str, message: str) -> str:
        """Process a user message and get AI response."""
        chat = self.chat_manager.get_chat(user_id, chat_id)
        if not chat:
            raise ValueError("Chat not found")
        
        # Add user message
        self.chat_manager.add_message(user_id, chat_id, "user", message)
        
        try:
            # TODO: Retrieve the conversation using the ChatManager
            
            # TODO: Use the OpenAI client to generate a response based on the conversation
            # - Set the model to "gpt-4"
            # - Pass the conversation as the messages parameter to provide context
            # - Use a temperature of 0.7 to balance creativity and coherence in responses
            # - Limit the response length to 500 tokens
            
            # TODO: Extract the AI's response from the OpenAI client response
            
            # TODO: Add the AI's response to the chat history
            
            # TODO: Return the AI's response
            
        except Exception as e:
            # TODO: Handle exceptions and raise a RuntimeError with an appropriate message

# main.py
from services.chat_service import ChatService

# Initialize the chat service
chat_service = ChatService()

# Simulate a user ID
user_id = "user123"

# Create a new chat session
chat_id = chat_service.create_chat(user_id)
print(f"Chat session created with ID: {chat_id}")

# Simulate sending a message
user_message = "How can I contact your company by email?"
print(f"User Message: {user_message}")

try:
    ai_response = chat_service.process_message(user_id, chat_id, user_message)
    print(f"AI Response: {ai_response}")
except Exception as e:
    print(f"Error: {e}")

```

The missing code to integrate the AI response generation using the OpenAI client and update the chat history has been added to the `ChatService` class.

Here is the completed `chat_service.py` file:

```python
#chat_service.py
import uuid
from openai import OpenAI
from models.chat import ChatManager

class ChatService:
    def __init__(self):
        self.chat_manager = ChatManager()
        # TODO: Initialize the OpenAI client to interact with the AI model
        self.openai_client = OpenAI()
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file."""
        try:
            with open(file_path, 'r') as f:
                return f.read()
        except Exception as e:
            print(f"Error loading system prompt: {e}")
            return "You are a helpful assistant."
    
    def create_chat(self, user_id: str) -> str:
        """Create a new chat session."""
        chat_id = str(uuid.uuid4())
        self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
        return chat_id
    
    def process_message(self, user_id: str, chat_id: str, message: str) -> str:
        """Process a user message and get AI response."""
        chat = self.chat_manager.get_chat(user_id, chat_id)
        if not chat:
            raise ValueError("Chat not found")
        
        # Add user message
        self.chat_manager.add_message(user_id, chat_id, "user", message)
        
        try:
            # TODO: Retrieve the conversation using the ChatManager
            conversation = self.chat_manager.get_conversation(user_id, chat_id)
            
            # TODO: Use the OpenAI client to generate a response based on the conversation
            # - Set the model to "gpt-4"
            # - Pass the conversation as the messages parameter to provide context
            # - Use a temperature of 0.7 to balance creativity and coherence in responses
            # - Limit the response length to 500 tokens
            completion = self.openai_client.chat.completions.create(
                model="gpt-4",
                messages=conversation,
                temperature=0.7,
                max_tokens=500
            )
            
            # TODO: Extract the AI's response from the OpenAI client response
            ai_response = completion.choices[0].message.content
            
            # TODO: Add the AI's response to the chat history
            self.chat_manager.add_message(user_id, chat_id, "assistant", ai_response)
            
            # TODO: Return the AI's response
            return ai_response
            
        except Exception as e:
            # TODO: Handle exceptions and raise a RuntimeError with an appropriate message
            raise RuntimeError(f"Failed to get AI response from OpenAI: {e}")

# main.py
from services.chat_service import ChatService

# Initialize the chat service
chat_service = ChatService()

# Simulate a user ID
user_id = "user123"

# Create a new chat session
chat_id = chat_service.create_chat(user_id)
print(f"Chat session created with ID: {chat_id}")

# Simulate sending a message
user_message = "How can I contact your company by email?"
print(f"User Message: {user_message}")

try:
    # NOTE: This part will fail if the OpenAI API key is not set up correctly 
    # in the environment (e.g., as OPENAI_API_KEY) and the 'models/chat.py' 
    # and 'data/system_prompt.txt' files are not present or correctly implemented 
    # to support the ChatManager and system prompt loading.
    ai_response = chat_service.process_message(user_id, chat_id, user_message)
    print(f"AI Response: {ai_response}")
except Exception as e:
    print(f"Error: {e}")
```

### Key Changes:

1.  **OpenAI Client Initialization:**
    ```python
    self.openai_client = OpenAI()
    ```
    in the `__init__` method.
2.  **Retrieve Conversation:**
    ```python
    conversation = self.chat_manager.get_conversation(user_id, chat_id)
    ```
3.  **Generate Response:**
    ```python
    completion = self.openai_client.chat.completions.create(
        model="gpt-4",
        messages=conversation,
        temperature=0.7,
        max_tokens=500
    )
    ```
    This uses the specified model, context, temperature, and token limit.
4.  **Extract Response:**
    ```python
    ai_response = completion.choices[0].message.content
    ```
5.  **Update History & Return:**
    ```python
    self.chat_manager.add_message(user_id, chat_id, "assistant", ai_response)
    return ai_response
    ```
6.  **Exception Handling:**
    ```python
    except Exception as e:
        raise RuntimeError(f"Failed to get AI response from OpenAI: {e}")
    ```

-----

Would you like to move on to the next step, perhaps focusing on **handling streaming responses** from the AI?

## Testing AI Context Retention

ou've successfully set up the chat service and processed the initial message. Now, let's enhance the AI's capability by testing its ability to retain context across multiple interactions. This will ensure the AI can handle multi-turn conversations effectively.

Here's what you need to do:

After sending the initial message, define a follow-up question that requires the AI to recall the context of the previous message. For example, you might ask: "Is there another way?" Print this follow-up question to the console.

Use the process_message method to send this follow-up question to the AI.

Print the AI's response to the follow-up question to verify its ability to maintain context across multiple messages.

This task will help you verify the AI's ability to maintain context across multiple messages. Keep going; you're doing well!

```python
# main.py
from services.chat_service import ChatService

# Initialize the chat service
chat_service = ChatService()

# Simulate a user ID
user_id = "user123"

# Create a new chat session
chat_id = chat_service.create_chat(user_id)
print(f"Chat session created with ID: {chat_id}")

# Define a user message
user_message = "How can I contact your company by email?"
print(f"User Message: {user_message}")

try:
    ai_response = chat_service.process_message(user_id, chat_id, user_message)
    print(f"AI Response: {ai_response}")
except Exception as e:
    print(f"Error: {e}")

# TODO: Define and print a follow-up question
# TODO: Send the follow-up question to test context retention
# TODO: Print the AI's response to the follow-up question

# chat_service.py 
import uuid
from openai import OpenAI
from models.chat import ChatManager

class ChatService:
    def __init__(self):
        self.chat_manager = ChatManager()
        self.openai_client = OpenAI()
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file."""
        try:
            with open(file_path, 'r') as f:
                return f.read()
        except Exception as e:
            print(f"Error loading system prompt: {e}")
            return "You are a helpful assistant."
    
    def create_chat(self, user_id: str) -> str:
        """Create a new chat session."""
        chat_id = str(uuid.uuid4())
        self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
        return chat_id
    
    def process_message(self, user_id: str, chat_id: str, message: str) -> str:
        """Process a user message and get AI response."""
        chat = self.chat_manager.get_chat(user_id, chat_id)
        if not chat:
            raise ValueError("Chat not found")
        
        # Add user message
        self.chat_manager.add_message(user_id, chat_id, "user", message)
        
        try:
            # Get AI response
            conversation = self.chat_manager.get_conversation(user_id, chat_id)
            
            response = self.openai_client.chat.completions.create(
                model="gpt-4",
                messages=conversation,
                temperature=0.7,
                max_tokens=500
            )
            
            ai_message = response.choices[0].message.content
            
            # Add AI response to chat history
            self.chat_manager.add_message(user_id, chat_id, "assistant", ai_message)
            
            return ai_message
            
        except Exception as e:
            raise RuntimeError(f"Error getting AI response: {str(e)}")

```

I've updated the `main.py` file with the follow-up question and the necessary calls to the `ChatService` to test **context retention**.

The key is that the `chat_service.process_message` uses the entire **conversation history** (which includes the initial question and the AI's first answer) to generate the response for the simple follow-up question, "Is there another way?".

Here is the completed `main.py`:

```python
# main.py
from services.chat_service import ChatService

# Initialize the chat service
chat_service = ChatService()

# Simulate a user ID
user_id = "user123"

# Create a new chat session
chat_id = chat_service.create_chat(user_id)
print(f"Chat session created with ID: {chat_id}")

# Define a user message
user_message = "How can I contact your company by email?"
print(f"\nUser Message: {user_message}")

try:
    ai_response = chat_service.process_message(user_id, chat_id, user_message)
    print(f"AI Response: {ai_response}")
except Exception as e:
    print(f"Error: {e}")

# --- Context Retention Test ---

# TODO: Define and print a follow-up question
follow_up_question = "Is there another way?"
print(f"\nFollow-up Question: {follow_up_question}")

try:
    # TODO: Send the follow-up question to test context retention
    follow_up_response = chat_service.process_message(user_id, chat_id, follow_up_question)
    
    # TODO: Print the AI's response to the follow-up question
    print(f"AI Follow-up Response (Context Check): {follow_up_response}")
    
except Exception as e:
    print(f"Error during follow-up: {e}")
```

### Explanation of Context Retention

The process works because:

1.  The `process_message` method calls `self.chat_manager.add_message(user_id, chat_id, "user", message)` to add the **new user message** to the history.
2.  It then calls `conversation = self.chat_manager.get_conversation(user_id, chat_id)` which retrieves **all previous messages** (including the system prompt, the first user message, and the first AI response).
3.  This **full conversation list** is passed as the `messages` parameter to the `openai_client.chat.completions.create` call, allowing the model to understand that "Is there another way?" relates to the **"contacting the company"** context established previously.

-----

Now that context retention is confirmed, would you like to explore how to **customize the AI's behavior** more effectively using the **system prompt**?

## Creating Multiple Chat Sessions

You've made great progress in setting up the ChatService class and processing user messages. Now, let's focus on creating multiple chat sessions for the same user to ensure that each session has a cleared context.

Here's what you need to do:

Create a Second Chat Session: Use the create_chat method to create a new chat session for the same user. This will ensure that the user can have multiple independent conversations.

Process a Message in the Second Chat: Simulate sending a message in the newly created chat session using the process_message method. This will test that the chat session is functioning correctly and independently from the first session.

This task will guide you through the process of creating and managing chat sessions, a fundamental aspect of building a chatbot service. Keep up the momentum!

```python
from services.chat_service import ChatService

# Initialize the chat service
chat_service = ChatService()

# Simulate a user ID
user_id = "user123"

# Create a new chat session
chat_id = chat_service.create_chat(user_id)
print(f"Chat session created with ID: {chat_id}")

# Simulate sending a message
user_message = "Hello, how are you?"

try:
    ai_response = chat_service.process_message(user_id, chat_id, user_message)
    print(f"AI Response: {ai_response}")
except Exception as e:
    print(f"Error: {e}")

# TODO: Create a second chat session for the same user and process a new message

import uuid
from openai import OpenAI
from models.chat import ChatManager

class ChatService:
    def __init__(self):
        self.chat_manager = ChatManager()
        self.openai_client = OpenAI()
        self.system_prompt = self.load_system_prompt('data/system_prompt.txt')
    
    def load_system_prompt(self, file_path: str) -> str:
        """Load the system prompt from file."""
        try:
            with open(file_path, 'r') as f:
                return f.read()
        except Exception as e:
            print(f"Error loading system prompt: {e}")
            return "You are a helpful assistant."
    
    def create_chat(self, user_id: str) -> str:
        """Create a new chat session."""
        chat_id = str(uuid.uuid4())
        self.chat_manager.create_chat(user_id, chat_id, self.system_prompt)
        return chat_id
    
    def process_message(self, user_id: str, chat_id: str, message: str) -> str:
        """Process a user message and get AI response."""
        chat = self.chat_manager.get_chat(user_id, chat_id)
        if not chat:
            raise ValueError("Chat not found")
        
        # Add user message
        self.chat_manager.add_message(user_id, chat_id, "user", message)
        
        try:
            # Get AI response
            conversation = self.chat_manager.get_conversation(user_id, chat_id)
            
            response = self.openai_client.chat.completions.create(
                model="gpt-4",
                messages=conversation,
                temperature=0.7,
                max_tokens=500
            )
            
            ai_message = response.choices[0].message.content
            
            # Add AI response to chat history
            self.chat_manager.add_message(user_id, chat_id, "assistant", ai_message)
            
            return ai_message
            
        except Exception as e:
            raise RuntimeError(f"Error getting AI response: {str(e)}")
```